# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We use the `mlcroissant` interface to inspect all record sets, fields, and their `@id` values. This information is useful for referencing during extraction and processing.

In [ ]:
# List all available record sets, fields, and columns - using @id for reference
print('Record sets:')
record_set_objs = list(dataset.record_sets())
for rec_set in record_set_objs:
    print(f"  @id: {rec_set['@id']}  name: {rec_set.get('name', '')}")

# For the first record set, print available fields (if record set exists)
if record_set_objs:
    example_rec_set_id = record_set_objs[0]['@id']
    print(f"\nFields for record set {example_rec_set_id}:")
    fields = dataset.fields(record_set=example_rec_set_id)
    for field in fields:
        print(f"  @id: {field['@id']}  name: {field.get('name', '')}  dataType: {field.get('dataType', '')}")

    print("\nPreview of some records in this record set:")
    for i, record in enumerate(dataset.records(record_set=example_rec_set_id)):
        print(record)
        if i >= 2:
            break

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from all record sets discovered in overview (by @id)
record_sets = [rec['@id'] for rec in dataset.record_sets()]
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f'Loaded DataFrame for record set {record_set_id} with shape {dataframes[record_set_id].shape}')
    else:
        print(f'No records found for record set {record_set_id}')

# Show columns for each DataFrame
for rs_id, df in dataframes.items():
    print(f'\nColumns for {rs_id}:')
    print(df.columns.tolist())
    print(df.head(2))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates:
- Filtering on a numeric field (e.g., number of unique peptides)
- Normalizing a numeric feature
- Grouping by a field such as protein accession or sample

In [ ]:
# Example: Perform EDA on the primary proteins record set
# Please adjust record set and field IDs based on printed output in previous cells

# By convention, pick the first non-empty dataframe for EDA
if dataframes:
    eda_rec_set_id = list(dataframes.keys())[0]
    df = dataframes[eda_rec_set_id].copy()
    print(f'Using record set: {eda_rec_set_id}')
    print(df.dtypes)
    print(df.head(3))

    # Pick a numeric field (use the printed columns and @id from previous cells)
    # Common names: e.g. 'Number of Peptides', 'MW [kDa]', or 'Normalized Abundance' fields
    # Try to heuristically guess numeric columns if types unknown
    numeric_candidates = [col for col in df.columns if df[col].dtype in [np.float64, np.int64, float, int] or np.issubdtype(df[col].dtype, np.number)]
    if not numeric_candidates:
        # Try to parse floats from object columns
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except Exception:
                continue
        numeric_candidates = [col for col in df.columns if df[col].dtype in [np.float64, np.int64, float, int] or np.issubdtype(df[col].dtype, np.number)]
    
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Selected numeric field: {numeric_field}")
    else:
        raise Exception("No numeric field found for EDA.")

    # Set threshold for filtering
    try:
        threshold = df[numeric_field].mean() if df[numeric_field].mean() > 0 else 1
    except:
        threshold = 1
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (count: {len(filtered_df)}):")
    print(filtered_df.head())

    # Normalize the field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field}:\n", filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Select a candidate group field
    group_field = None
    for possible in ['Sample', 'sample', 'Protein', 'protein', 'Gene Symbol', 'gene_symbol', 'gene', 'Accession', 'accession']:
        if possible in df.columns:
            group_field = possible
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by '{group_field}', showing head:")
        print(grouped_df.head())
else:
    print('No tabular dataframes loaded for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. This example shows a histogram of the numeric field distribution, and a scatterplot of normalized vs. original values if available.

In [ ]:
if dataframes:
    eda_rec_set_id = list(dataframes.keys())[0]
    df = dataframes[eda_rec_set_id]
    # Try to use the same numeric field as before
    numeric_candidates = [col for col in df.columns if df[col].dtype in [np.float64, np.int64, float, int] or np.issubdtype(df[col].dtype, np.number)]
    if not numeric_candidates:
        # Try to parse floats
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except Exception:
                pass
        numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_candidates:
        field = numeric_candidates[0]
        plt.figure(figsize=(8, 4))
        df[field].hist(bins=30)
        plt.title(f'Distribution of: {field}')
        plt.xlabel(field)
        plt.ylabel('Frequency')
        plt.show()
        # If normalized available, make a scatter
        norm_col = f"{field}_normalized"
        if norm_col in df.columns:
            plt.figure(figsize=(6,4))
            plt.scatter(df[field], df[norm_col], alpha=0.5)
            plt.xlabel(field)
            plt.ylabel(norm_col)
            plt.title(f'Original vs normalized: {field}')
            plt.show()

## 6. Conclusion
This notebook demonstrated loading and exploring the FAIR<sup>2</sup> dataset describing mass spectrometry analysis using the `mlcroissant` library. We:
- Inspected the dataset schema and record set IDs
- Loaded data into pandas DataFrames
- Filtered and normalized numerical attributes
- Grouped by meaningful categorical identifiers
- Visualized key numeric distributions

Refer to the printed output and schema documentation for authoritative `@id` references for each data element and for extending the analysis to additional record sets and fields.